## Подбор лучший параметров TF-IDF + Logreg (RandomizedSearchCV) - Kaggle

In [ ]:
# ============================================
# БЛОК 1. Загрузка датасета Sentiment140 с Kaggle
# ============================================

from google.colab import files

import os
import shutil
import zipfile
import re

import pandas as pd


# 1. Загружаем kaggle.json
# Перед запуском скачай kaggle.json:
# Kaggle → Account → API → Create New Token
uploaded = files.upload()  # выбери kaggle.json


# 2. Настраиваем Kaggle API
os.makedirs("/root/.kaggle", exist_ok=True)

if "kaggle.json" in uploaded:
    shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")

os.chmod("/root/.kaggle/kaggle.json", 600)


# 3. Скачиваем датасет
os.makedirs("./data", exist_ok=True)

!kaggle datasets download -d kazanova/sentiment140 -p ./data


# 4. Распаковываем архив
zip_path = "./data/sentiment140.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("./data")

print("Файлы в ./data:")
print(os.listdir("./data"))


# 5. Загружаем CSV
csv_path = "./data/training.1600000.processed.noemoticon.csv"

feature_names = ["target", "id", "date", "flag", "user", "text"]

df = pd.read_csv(
    csv_path,
    sep=",",
    encoding="latin-1",
    names=feature_names
)

print("Размер исходного датасета:", df.shape)
display(df.head())


# 6. Функция очистки текста
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)        # убрать ссылки
    text = re.sub(r"@\w+", "", text)           # убрать @username
    text = re.sub(r"[^a-z\s]", "", text)      # оставить только английские буквы и пробелы
    text = re.sub(r"\s+", " ", text).strip()  # убрать лишние пробелы
    return text


# 7. Кодируем целевую переменную:
# 0 → 0 негатив
# 4 → 1 позитив
df["target"] = df["target"].map({0: 0, 4: 1})

# Удаляем возможные пустые значения
df = df.dropna(subset=["target", "text"])

# Создаём очищенный текст
df["text_clean"] = df["text"].apply(clean_text)

print("Размер после обработки:", df.shape)
print(df["target"].value_counts())
display(df.sample(5, random_state=42))

In [ ]:
# ============================================
# БЛОК 2. Обучение модели + вывод результата
# ============================================

import time
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


start = time.time()


# 1. Выбор размера выборки
# Для первого запуска в Colab лучше взять часть данных,
# чтобы RandomizedSearchCV не работал слишком долго.
SAMPLE_SIZE = 500_000

df_work = df.sample(n=SAMPLE_SIZE, random_state=42)

# Если хочешь обучать на всём датасете, замени блок выше на:
# df_work = df.copy()

print("Размер рабочей выборки:", df_work.shape)
print(df_work["target"].value_counts(normalize=True))

# 2. Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    df_work["text_clean"],
    df_work["target"],
    stratify=df_work["target"],
    test_size=0.3,
    random_state=1,
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


# 3. Pipeline: TF-IDF + LogisticRegression
pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer(sublinear_tf=True)),
        ("logreg", LogisticRegression(max_iter=1000, solver="saga", n_jobs=-1)),
    ]
)


# 4. Параметры для RandomizedSearchCV
param_distributions = [
    {
        "tfidf__analyzer": ["word"],
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "tfidf__min_df": [2, 3, 5],
        "tfidf__max_features": [50_000, 100_000, 200_000],
        "logreg__C": [0.3, 0.5, 1.0, 2.0, 3.0],
        "logreg__penalty": ["l2"],
    },
    {
        "tfidf__analyzer": ["char_wb"],
        "tfidf__ngram_range": [(3, 5), (4, 6), (5, 7)],
        "tfidf__min_df": [3, 5, 7],
        "tfidf__max_features": [50_000, 100_000],
        "logreg__C": [0.5, 1.0, 2.0, 3.0],
        "logreg__penalty": ["l2"],
    },
]


# 5. Поиск лучших параметров
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=10,  # для Colab лучше начать с 10
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=2,
    random_state=42,
)

search.fit(X_train, y_train)


# 6. Получаем лучшую модель
best_model = search.best_estimator_


# 7. Вывод лучших параметров
print("\nЛучшие параметры:")
print(search.best_params_)

print(f"\nЛучшая accuracy на кросс-валидации: {search.best_score_:.4f}")


# 8. Проверка на тестовой выборке
y_pred = best_model.predict(X_test)

print("\nРезультат на тестовой выборке:")
print(classification_report(y_test, y_pred))
print(f"Accuracy test: {accuracy_score(y_test, y_pred):.4f}")

print(f"\nВремя выполнения: {time.time() - start:.2f} сек.")


# 9. Функция для проверки своего текста
def predict_sentiment(text):
    clean = clean_text(text)

    pred = best_model.predict([clean])[0]
    proba = best_model.predict_proba([clean])[0]

    label = "Позитивное настроение" if pred == 1 else "Негативное настроение"

    result = {
        "text": text,
        "clean_text": clean,
        "prediction": label,
        "negative_probability": round(float(proba[0]), 4),
        "positive_probability": round(float(proba[1]), 4),
    }

    return result


# 10. Примеры проверки
print("\nПроверка модели:")
print(predict_sentiment("I feel happy and calm today"))
print(predict_sentiment("I am so tired and sad today"))

# {
#     "tfidf__ngram_range": (1, 2),
#     "tfidf__min_df": 3,
#     "tfidf__max_features": 200000,
#     "tfidf__analyzer": "word",
#     "logreg__penalty": "l2",
#     "logreg__C": 1.0,
# }

{'tfidf__ngram_range': (1, 2), 'tfidf__min_df': 3, 'tfidf__max_features': 200000, 'tfidf__analyzer': 'word', 'logreg__penalty': 'l2', 'logreg__C': 1.0}

In [ ]:
# ============================================
# БЛОК 3. Сохранение модели
# ============================================

import os
import joblib

from google.colab import files


# 1. Создаём папку для моделей
os.makedirs("./models", exist_ok=True)


# 2. Путь для сохранения модели
model_path = "./models/sentiment_tfidf_logreg.pkl"


# 3. Сохраняем лучшую модель
joblib.dump(best_model, model_path)

print(f"Модель сохранена: {model_path}")


# 4. Скачиваем модель на компьютер
files.download(model_path)